# ONNX object detection wrapper

This notebook shows the minimal path for loading a JATIC_ONNX object detection model with `OnnxODModel`. The small model used here is deterministic and returns two fixed boxes so the example is lightweight and repeatable.

The wrapper requires the optional ONNX dependencies (`pip install "checkmaite[onnx]"`). It expects a JATIC_ONNX metadata JSON file with `interface`, `io`, and `index2label` fields. See the developer notes in `docs/development/onnx_model_wrappers.md` for implementation details.

In [ ]:
from pathlib import Path

import numpy as np

from checkmaite.core.object_detection.models import OnnxODModel


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "tests" / "data_for_tests" / "jatic_onnx_od").exists():
            return candidate
    raise FileNotFoundError("Could not find tests/data_for_tests/jatic_onnx_od")


repo_root = find_repo_root(Path.cwd())
fixture_dir = repo_root / "tests" / "data_for_tests" / "jatic_onnx_od"
model_path = fixture_dir / "constant_detector.onnx"
metadata_path = fixture_dir / "model-metadata.json"

## Load the model

`model_weights_path` in the programmatic loader corresponds to the `.onnx` file. `model_config_path` corresponds to the JATIC_ONNX metadata file. Direct wrapper construction uses the same two paths as `weights_path` and `config_path`.

In [ ]:
model = OnnxODModel(
    weights_path=model_path,
    config_path=metadata_path,
    device="cpu",
)

model.metadata

## Run inference

The wrapper accepts a sequence of CHW images. The fixture metadata declares a fixed `10 x 20` RGB input, so this example creates one zero-valued image with shape `(3, 10, 20)`. JATIC_ONNX boxes are normalized in the ONNX output; `OnnxODModel` converts them back to pixel-coordinate `xyxy` boxes in the returned `DetectionTarget`.

In [ ]:
image = np.zeros((3, 10, 20), dtype=np.uint8)
prediction = model([image])[0]

print("boxes:")
print(prediction.boxes)
print("labels:", prediction.labels)
print("scores:", prediction.scores)

## Loading through `load_models`

The dashboard and programmatic configuration path use the task module's `load_models` dispatch. For ONNX, use `model_type="jatic_onnx"`.

In [ ]:
from checkmaite.core.object_detection.models import load_models

loaded = load_models(
    {
        "fixture_detector": {
            "model_type": "jatic_onnx",
            "model_weights_path": model_path,
            "model_config_path": metadata_path,
        }
    },
    device="cpu",
)

loaded["fixture_detector"]([image])[0].labels